# 3D Mesh Quality Control - Solution
This notebook trains an EfficientNet-B0 model on the 2D multi-view renders to predict 10 defect classes. It then infers the overall quality label and creates `submission.csv`.
It is designed to run in Google Colab.

In [ ]:
!pip install timm scikit-learn pandas numpy Pillow tqdm

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm
from sklearn.metrics import f1_score
from tqdm import tqdm

# Set seeds for reproducibility
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)

In [ ]:
# Config
class Config:
    # Adjust these paths if your data is in Google Drive or has different names (e.g. 'train (2)/train')
    TRAIN_CSV = 'data/train.csv'
    TEST_CSV = 'data/test.csv'

    # Path untuk membaca gambar dari memori lokal Colab yang baru saja diekstrak
    # Catatan: Jika setelah diekstrak Colab bilang "file not found",
    # coba ubah TRAIN_IMG_DIR menjadi 'data/train (2)/train'
    # (Tergantung struktur asli di dalam file zip Anda)
    TRAIN_IMG_DIR = 'data/train'
    TEST_IMG_DIR = 'data/test'


    MODEL_NAME = 'efficientnet_b0'
    IMG_SIZE = (384, 576) # (height, width) downscaled from 1024x1536
    BATCH_SIZE = 16
    EPOCHS = 10
    LR = 1e-3
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    DEFECTS = ['abstract','artifacts','intersection','lowpoly','noisy','open','partial','scale','set','simple']

In [ ]:
class MeshDataset(Dataset):
    def __init__(self, df, img_dir, transforms=None, is_test=False):
        self.df = df
        self.img_dir = img_dir
        self.transforms = transforms
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        item_id = row['item_id']

        # Load stitched image
        img_path = os.path.join(self.img_dir, f"{item_id}.png")
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            # Fallback if image not found
            image = Image.new('RGB', (1536, 1024))

        if self.transforms:
            image = self.transforms(image)

        if self.is_test:
            return image, item_id
        else:
            labels = row[Config.DEFECTS].values.astype(np.float32)
            return image, torch.tensor(labels)

train_transform = transforms.Compose([
    transforms.Resize(Config.IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(Config.IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# Load data
try:
    train_df = pd.read_csv(Config.TRAIN_CSV)
except:
    # create dummy for testing notebook creation
    train_df = pd.DataFrame(columns=['item_id'] + Config.DEFECTS + ['quality'])

# Split into train and validation (80/20)
train_df_shuffled = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
split_idx = int(0.8 * len(train_df_shuffled))
train_data = train_df_shuffled.iloc[:split_idx]
val_data = train_df_shuffled.iloc[split_idx:]

train_dataset = MeshDataset(train_data, Config.TRAIN_IMG_DIR, train_transform)
val_dataset = MeshDataset(val_data, Config.TRAIN_IMG_DIR, val_transform)

train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
class MeshQualityModel(nn.Module):
    def __init__(self, model_name=Config.MODEL_NAME, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=len(Config.DEFECTS))

    def forward(self, x):
        return self.backbone(x)

model = MeshQualityModel().to(Config.DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=Config.LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=Config.EPOCHS)

In [ ]:
def get_f1_score(y_true, y_pred_logits):
    y_pred = (torch.sigmoid(y_pred_logits) > 0.5).int().cpu().numpy()
    y_true = y_true.cpu().numpy()
    return f1_score(y_true, y_pred, average='weighted')

best_f1 = 0
for epoch in range(Config.EPOCHS):
    model.train()
    train_loss = 0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{Config.EPOCHS} [Train]"):
        images, labels = images.to(Config.DEVICE), labels.to(Config.DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{Config.EPOCHS} [Val]"):
            images, labels = images.to(Config.DEVICE), labels.to(Config.DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            all_preds.append(outputs)
            all_labels.append(labels)

    scheduler.step()

    if all_preds:
        all_preds = torch.cat(all_preds)
        all_labels = torch.cat(all_labels)
        val_f1 = get_f1_score(all_labels, all_preds)
    else:
        val_f1 = 0

    print(f"Epoch {epoch+1} - Train Loss: {train_loss/len(train_loader):.4f} - Val Loss: {val_loss/max(1, len(val_loader)):.4f} - Val F1: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), 'outputs/best_model.pth')
        print("Saved new best model.")

## 2. Inference & Submission Generation

In [ ]:
try:
    model.load_state_dict(torch.load('best_model.pth'))
    model.eval()
except:
    pass # Handle if training was skipped

try:
    test_df = pd.read_csv(Config.TEST_CSV)
    test_dataset = MeshDataset(test_df, Config.TEST_IMG_DIR, val_transform, is_test=True)
    test_loader = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=2)

    test_preds = []
    item_ids = []
    with torch.no_grad():
        for images, ids in tqdm(test_loader, desc="Predicting Test"):
            images = images.to(Config.DEVICE)
            outputs = model(images)
            preds = (torch.sigmoid(outputs) > 0.5).int().cpu().numpy()
            test_preds.extend(preds)
            item_ids.extend(ids)

    # Create submission
    sub_df = pd.DataFrame(test_preds, columns=Config.DEFECTS)
    sub_df.insert(0, 'item_id', item_ids)

    # Calculate quality: 1 if sum of defects == 0, else 0
    sub_df['quality'] = (sub_df[Config.DEFECTS].sum(axis=1) == 0).astype(int)

    # Ensure correct column order
    cols = ['item_id'] + Config.DEFECTS + ['quality']
    sub_df = sub_df[cols]
    sub_df.to_csv('outputs/submission.csv', index=False)
    print("Submission saved to submission.csv")
except Exception as e:
    print("Inference error:", e)